# RAG

In [3]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

df = pd.read_csv('../data/5k_resenias.csv', sep=';')
df = df[df['review_text'].notna()]
df["resena_id"] = np.arange(1, len(df) + 1)
print(len(df))
df.head(3)


5089


,business_name,total_reviews,review_rating,review_text,datetime_utc,pais,resena_id
0,Visita guiada por el Museo del Prado y el Pala...,1647,5,"La experiencia fue excelente, desde mucho ante...",07/08/2026,ESP,1
1,Visita guiada por el Museo del Prado y el Pala...,1647,5,Nuestro guía Juan Antonio tenía una gran exper...,07/02/2026,ESP,2
2,Visita guiada por el Museo del Prado y el Pala...,1647,5,"El tour comenzó en el Palacio Real, siguiendo ...",05/30/2026,ESP,3


## Chunking

In [ ]:
def chunking_fijo(texto, tamano_chunk=300, overlap=50):
  """Divide por caracteres con solapamiento."""
  chunks = []
  inicio = 0
  while inicio < len(texto):
    fin = inicio + tamano_chunk
    chunk = texto[inicio:fin].strip()
    if chunk:
      chunks.append(chunk)
    inicio = fin - overlap
  return chunks


def chunking_oraciones(texto, oraciones_por_chunk=3, overlap_oraciones=1):
  """Divide agrupando N oraciones."""
  oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
  oraciones = [o.strip() for o in oraciones if o.strip()]
  chunks = []
  i = 0
  while i < len(oraciones):
    grupo = oraciones[i : i + oraciones_por_chunk]
    chunk = ' '.join(grupo)
    if chunk:
      chunks.append(chunk)
    i += oraciones_por_chunk - overlap_oraciones
  return chunks


def chunking_parrafos(texto, min_longitud=50):
  """Divide por párrafos (saltos de línea) combinando los cortos[cite: 1]."""
  parrafos = re.split(r'\n\s*\n', texto)
  parrafos = [p.strip() for p in parrafos if p.strip()]
  chunks = []
  buffer = ''
  for p in parrafos:
    if len(buffer) + len(p) < min_longitud * 3:
      buffer += ' ' + p
    else:
      if buffer.strip():
        chunks.append(buffer.strip())
      buffer = p
  if buffer.strip():
    chunks.append(buffer.strip())
  return chunks

In [ ]:
df['chunks_fijo'] = df['review_text'].apply(
    lambda x: chunking_fijo(x, tamano_chunk=120, overlap=30)
)
df['chunks_oraciones'] = df['review_text'].apply(
    lambda x: chunking_oraciones(x, oraciones_por_chunk=2, overlap_oraciones=1)
)
df['chunks_parrafos'] = df['review_text'].apply(chunking_parrafos)

estrategias = {
    'fijo': 'chunks_fijo',
    'oraciones': 'chunks_oraciones',
    'parrafos': 'chunks_parrafos',
}

In [6]:
for nombre, col in [
    ('Fijo', 'chunks_fijo'),
    ('Oraciones', 'chunks_oraciones'),
    ('Párrafos', 'chunks_parrafos'),
]:
  todos_los_chunks = [c for lista in df[col] for c in lista]
  tamanos = [len(c) for c in todos_los_chunks]
  print(f'Estrategia {nombre}:')
  print(f'  Total chunks generados: {len(todos_los_chunks)}')
  print(f'  Tamaño promedio: {np.mean(tamanos):.0f} caracteres')
  print(f'  Min/Max: {min(tamanos)}/{max(tamanos)} caracteres\n')

Estrategia Fijo:
  Total chunks generados: 10800
  Tamaño promedio: 83 caracteres
  Min/Max: 1/120 caracteres

Estrategia Oraciones:
  Total chunks generados: 10861
  Tamaño promedio: 103 caracteres
  Min/Max: 1/1483 caracteres

Estrategia Párrafos:
  Total chunks generados: 5089
  Tamaño promedio: 146 caracteres
  Min/Max: 3/3203 caracteres



## Embeddings

In [7]:
modelo_e5 = SentenceTransformer('intfloat/multilingual-e5-small')


In [ ]:
dict_embeddings_guardar = {}


for nombre_est, columna_chunk in estrategias.items():
  registros_chunks = []
  for _, row in df.iterrows():
    for idx_chunk, texto_chunk in enumerate(row[columna_chunk]):
      registros_chunks.append({
          'resena_id': row['resena_id'],
          'chunk_index': idx_chunk,
          'texto': texto_chunk,
      })

  df_chunks = pd.DataFrame(registros_chunks)


  # Para modelos E5 es buena práctica agregar el prefijo 'passage: ' antes de codificar
  textos_e5 = [f'passage: {t}' for t in df_chunks['texto']]

  embeddings = modelo_e5.encode(textos_e5, show_progress_bar=True)

  df_chunks.to_csv(f'metadata_chunks_{nombre_est}.csv', index=False)
  np.save(f'embeddings_{nombre_est}.npy', embeddings)

In [4]:
embeddings_oraciones_cargados = np.load('embeddings_oraciones.npy')
metadata_oraciones_cargada = pd.read_csv('metadata_chunks_oraciones.csv')

print(
    '\nEmbeddings:'
    f' Formato: {embeddings_oraciones_cargados.shape}'
)


Embeddings: Formato: (10861, 384)


## Busqueda Semantica con FAISS

In [5]:
def buscar_en_estrategia(
    pregunta, nombre_estrategia, modelo, top_k=2
):
  embeddings = np.load(f'embeddings_{nombre_estrategia}.npy')
  metadata = pd.read_csv(f'metadata_chunks_{nombre_estrategia}.csv')

  dimension = embeddings.shape[1]
  indice = faiss.IndexFlatL2(dimension) 

  faiss.normalize_L2(embeddings)  
  indice.add(embeddings)

  embedding_pregunta = modelo.encode([f'query: {pregunta}'])
  faiss.normalize_L2(embedding_pregunta) 

  distancias, indices = indice.search(embedding_pregunta, top_k)

  resultados = []
  for dist, idx in zip(distancias[0], indices[0]):
    row = metadata.iloc[idx]
    resultados.append({
        'resena_id': row['resena_id'],
        'chunk': row['texto'],
        'score': float(1 - dist),  # Convertir distancia a similitud[cite: 1]
    })

  return resultados

In [8]:
pregunta_prueba = '¿Las atracciones estan limpias?'
estrategias = ['fijo', 'oraciones', 'parrafos']

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Las atracciones estan limpias?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 1628 | Score: 0.7685
    Texto: 'as, pero no muy limpias, sin embargo es de entender por la vegetación del lugar. En cuanto a atracciones hay 3, me encan'
[2] Reseña ID: 2364 | Score: 0.7566
    Texto: 'ho de que a pesar de que en general el mantenimiento del lugar está bien, algunas de las atracciones sí les falta manten'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 1573 | Score: 0.7678
    Texto: 'Al menos limpieza, es una lastima ya que es bonito, pero especialmente el area de juegos de niños esta sumamente sucia y es desagradable.'
[2] Reseña ID: 2101 | Score: 0.7665
    Texto: 'Muchas atracciones, muy amplio, muy limpio.'

--- ESTRATEGIA: PARRAFOS ---
[1] Reseña ID: 2353 | Score: 0.7547
    Texto: 'Un lugar maravilloso para los niños. Hay que llegar temprano para tener tiempo de recorrer todo. Sin embargo el mantenimiento deja mucho que desear. Las atracciones están cada vez más deterio

In [15]:
pregunta_prueba = '¿Cuanto dura el tour del Museo del Prado?'

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Cuanto dura el tour del Museo del Prado?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 393 | Score: 0.8071
    Texto: '5 horas no son suficientes. La calidad de las obras que alberga el Museo El Prado es impresionante. Y si la visita va ac'
[2] Reseña ID: 241 | Score: 0.8064
    Texto: 'Es un Tour un poco ambicioso, tanto el Palacio Real como el Museo del Prado merecen más tiempo. El guía Martin excepcion'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 47 | Score: 0.8193
    Texto: 'Es un tour largo, terminamos sobre las 16:30, pero merece cada minuto. Al terminar en el Prado tienes la opción de quedarte más tiempo por tu cuenta al acabar el tour.'
[2] Reseña ID: 36 | Score: 0.7985
    Texto: 'Elegimos el tour combinado Palacio Real + Prado, un tour de cinco horas andando (!) que se le pasarán volando si tienen la suerte de tener un guía como Ángel. Claro que el Prado dá para varios días, pero en poco más de dos horas Ángel les proporcionará una intensa inmersión en un

In [20]:
pregunta_prueba = '¿Angel es un buen guia?'

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Angel es un buen guia?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 104 | Score: 0.8157
    Texto: 'Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos. Su sentido del humor hace que la visita sea m'
[2] Reseña ID: 513 | Score: 0.8095
    Texto: 'Angel es un guía a quien realmente le apasiona lo que hace. Le interesa que aprendamos y hacernos vivir una experiencia'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 33 | Score: 0.8162
    Texto: 'El guía, Angel, es una maravilla, para la actividad tan densa que tiene que realizar es capaz de mantener el interés del grupo además de su propio entusiasmo por enseñar. Solo por poner una pega, estaría bien dedicar la mañana a un monumento y la tarde a otro dejando un descanso en medio para comer y descansar un rato.'
[2] Reseña ID: 104 | Score: 0.8120
    Texto: 'Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos.'

--- ESTRATEGIA: PARRAFOS ---
[1] Reseña ID: 104 | Score: 0.8191
    

In [21]:
pregunta_prueba = '¿Hay animales en el volcan?'

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Hay animales en el volcan?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 1237 | Score: 0.7632
    Texto: 'Muy recomendado el tour para ir a las faldas del volcán, en el camino hay diversos animales como monos capuchino o tucan'
[2] Reseña ID: 1309 | Score: 0.7203
    Texto: 'ciles de recorrer, no se aprecian muchos animales y desde donde terminan todos los senderos está muy lejos del volcán, u'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 1237 | Score: 0.7496
    Texto: 'Muy recomendado el tour para ir a las faldas del volcán, en el camino hay diversos animales como monos capuchino o tucanes. La cascada y río es hermoso y refrescante.'
[2] Reseña ID: 1300 | Score: 0.7312
    Texto: 'La reserva está dividida en dos secciones, la 1º es la sección volcán y la 2ª sección, es la península, hay muchos animales llamados “pisotes” que son muy amigables, pero los guardias especifican a la hora de entrar que no le puedes dar de comer a ninguno de los animales en la reserva.'



## Generador

In [14]:
import ollama

# Reutilizamos la función de búsqueda que creamos en el paso anterior
# Búsqueda semántica usando la mejor estrategia (ej. 'oraciones')
pregunta_usuario = "¿Angel es un buen guia?"
estrategia_elegida = "oraciones"

# 1. Recuperar los chunks más relevantes desde FAISS[cite: 1]
resultados_faiss = buscar_en_estrategia(
    pregunta=pregunta_usuario,
    nombre_estrategia=estrategia_elegida,
    modelo=modelo_e5,
    top_k=3,
)  #[cite: 1]

# 2. Unir el texto de los fragmentos recuperados para formar el contexto[cite: 1]
contexto_recuperado = "\n\n".join([r["chunk"] for r in resultados_faiss])

print("--- CONTEXTO RECUPERADO DE LA BASE DE DATOS ---")
print(contexto_recuperado)
print("------------------------------------------------\n")


# 3. Definir la función que interactúa con Qwen 2.5 7B vía Ollama
def responder_con_qwen(pregunta, contexto, modelo_ollama="qwen2.5:7b"):
  # System prompt estructurado para acotar las respuestas al contexto
  prompt_sistema = (
      "Eres un asistente virtual especializado en análisis de reseñas"
      " turísticas.\nResponde a la pregunta del usuario utilizando"
      " ÚNICAMENTE la información proporcionada en el Contexto.\nSi la"
      " respuesta no está en el contexto, di 'No dispongo de suficiente"
      " información en las reseñas para responder'."
  )

  prompt_usuario = (
      f"Contexto relevante:\n{contexto}\n\nPregunta: {pregunta}\n\nRespuesta:"
  )

  # Llamada local a Ollama
  response = ollama.chat(
      model=modelo_ollama,
      messages=[
          {"role": "system", "content": prompt_sistema},
          {"role": "user", "content": prompt_usuario},
      ],
      options={
          "temperature": 0.2  # Temperatura baja para evitar alucinaciones
      },
  )

  return response["message"]["content"]  #


# 4. Generar la respuesta contextualizada
respuesta_final = responder_con_qwen(
    pregunta=pregunta_usuario, contexto=contexto_recuperado
)

print(f"PREGUNTA: {pregunta_usuario}\n")
print(f"RESPUESTA DE QWEN 2.5 7B:\n{respuesta_final}")

--- CONTEXTO RECUPERADO DE LA BASE DE DATOS ---
El guía, Angel, es una maravilla, para la actividad tan densa que tiene que realizar es capaz de mantener el interés del grupo además de su propio entusiasmo por enseñar. Solo por poner una pega, estaría bien dedicar la mañana a un monumento y la tarde a otro dejando un descanso en medio para comer y descansar un rato.

Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos.

Angel es un guía simpático pero con muy poco conocimiento histórico. Habla muy poco de historia y mucho de cosas irrelevantes y repetitivas.
------------------------------------------------

PREGUNTA: ¿Angel es un buen guia?

RESPUESTA DE QWEN 2.5 7B:
Sí, Angel es un buen guía. Algunas reseñas lo describen como un guía fantástico, atento, agradable y con muchos conocimientos. Sin embargo, también se menciona que en algunas reseñas se considera que tiene poco conocimiento histórico y habla mucho de cosas irrelevantes y repetitivas. A pesar de esto, 